[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch12-benchmarking/03-roofline-mo-rong-va-thoi-gian-dat-do-chinh-xac.ipynb)

# Mái nhà, mở rộng và thời gian đạt độ chính xác: benchmark hệ thống từ A100 tới CPU của bạn

Chương 12 định nghĩa benchmark hệ thống là phép đo giữ nguyên tải và mục tiêu chất lượng, rồi đổi phần
cứng và phần mềm. Thứ nó đo là hiệu suất phần cứng, tức mức duy trì chia cho mức đỉnh. Ba phép tính
nhẩm của chương nói con số ấy đến từ đâu: mật độ tính toán (arithmetic intensity) đặt một tải dưới mái
nhà của biểu đồ roofline, phụ trội truyền thông ăn mất một phần khi thêm GPU, và tốc độ thô không có
nghĩa nếu mô hình không chạm tới mục tiêu chất lượng.

Phần cứng lớn ở đây (A100, H100, cụm 8 GPU) chỉ được **tính** từ số của sách. Phần đo được chạy trên
CPU của bạn, và những con số đo ấy không được đặt cạnh số của sách.

**Bạn sẽ làm:**
1. tính khoảng cách giữa đỉnh và mức duy trì của A100, và mức sử dụng của câu hỏi trong checkpoint;
2. dựng lại phép tính nhẩm 1.2 từ một BERT-Base dựng bằng PyTorch, rồi quét mật độ tính toán theo batch
   và độ dài chuỗi;
3. đo roofline của chính CPU của bạn, và xem vài kernel nằm ở đâu dưới mái;
4. tính lại phép tính nhẩm 1.3 về hiệu suất mở rộng (scaling efficiency), và xem vì sao ngoại suy tuyến
   tính sai;
5. đo thời gian đạt độ chính xác (time-to-accuracy) của bốn cấu hình huấn luyện trên MNIST.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Đỉnh và mức duy trì

Định nghĩa 1.1 của sách đặt hai mức cạnh nhau: A100 đạt 312 TFLOP/s ở BF16 khi ở đỉnh, nhưng trong một
kịch bản minh hoạ với mức sử dụng FLOP của mô hình (Model FLOPs Utilization, MFU) từ 30 tới 50 phần
trăm, nó chỉ duy trì được một phần. Phần còn lại mất vào chờ bộ nhớ, bong bóng pipeline (pipeline
bubble) và phụ trội khởi chạy kernel.

**Dự đoán:** ở MFU 30 phần trăm, đỉnh gấp mức duy trì bao nhiêu lần?

In [ ]:
DINH_BF16 = sach(312, nguon="312 TFLOP/s (BF16) · 30–50 phần trăm")  # TFLOP/s
mfu_thap = sach(30, nguon="312 TFLOP/s (BF16) · 30–50 phần trăm") / 100
mfu_cao = sach(50, nguon="312 TFLOP/s (BF16) · 30–50 phần trăm") / 100
theo_sach("mức duy trì khi MFU 30 % (TFLOP/s)", DINH_BF16 * mfu_thap, sach=93.6, nguon="93.6–156 TFLOP/s")
theo_sach("mức duy trì khi MFU 50 % (TFLOP/s)", DINH_BF16 * mfu_cao, sach=156, nguon="93.6–156 TFLOP/s")
theo_sach("đỉnh gấp mức duy trì, MFU 50 %", 1 / mfu_cao, sach=2, nguon="≈2–3.3×", tol=0.05)
theo_sach("đỉnh gấp mức duy trì, MFU 30 %", 1 / mfu_thap, sach=3.3, nguon="≈2–3.3×", tol=0.05)

Checkpoint 1.1 hỏi ngược lại: một tuyên bố đỉnh 1,000 TFLOP/s, một benchmark duy trì 350 TFLOP/s, vậy MFU
là bao nhiêu, và riêng tỉ lệ ấy có chỉ ra điểm nghẽn không? Quy tắc kiểm tra tốc độ ánh sáng của góc nhìn
1.4 cũng là một phép chia như thế: một kernel đạt 10 TFLOP/s trên H100, với đỉnh khoảng 989 TFLOP/s ở FP16
và khoảng 1,979 TFLOP/s ở FP8 dày.

In [ ]:
dinh_qc = sach(1_000, nguon="1,000 · 350 TFLOP/s")
duy_tri = sach(350, nguon="1,000 · 350 TFLOP/s")
theo_sach("MFU (%)", 100 * duy_tri / dinh_qc, sach=35, nguon="350 TFLOP/s · 35 phần trăm")

kernel = sach(10, nguon="10 TFLOP/s · ≈989", tol=0.5)
h100_fp16 = sach(989, nguon="≈989 · ≈1,979 TFLOP/s", tol=0.5)
h100_fp8 = sach(1_979, nguon="≈989 · ≈1,979 TFLOP/s", tol=0.5)
print(f"kernel 10 TFLOP/s dùng {100 * kernel / h100_fp16:.2f} % đỉnh FP16 của H100,"
      f" và {100 * kernel / h100_fp8:.2f} % đỉnh FP8 dày")

Phép chia cho 35 phần trăm, nhưng không nói 65 phần trăm còn lại mất vào đâu. Một tải nghẽn ở bộ nhớ
(memory-bound) và một tải bị phụ trội điều phối (dispatch) chặn có thể cho cùng một tỉ lệ. Muốn phân biệt
hai trường hợp, phải đo thêm băng thông (bandwidth), và phần 3 sẽ đo cả hai trường hợp ấy trên CPU.

## 2. Roofline của A100 cho BERT

Biểu đồ roofline có hai trần. Hiệu năng đạt được không vượt đỉnh tính toán, và cũng không vượt băng
thông nhân với mật độ tính toán, tức số FLOP làm được trên mỗi byte đọc ghi bộ nhớ. Hai trần gặp nhau ở
điểm gãy (ridge point). Tải có mật độ dưới điểm gãy thì nghẽn ở bộ nhớ, còn trên điểm gãy thì nghẽn ở
tính toán (compute-bound).

In [ ]:
DINH = sach(312, nguon="312 · 19.5 TFLOP/s · 2.04 TB/s")       # TFLOP/s, FP16 Tensor Core
DINH_FP32 = sach(19.5, nguon="312 · 19.5 TFLOP/s · 2.04 TB/s")  # TFLOP/s
BW = sach(2.04, nguon="312 · 19.5 TFLOP/s · 2.04 TB/s")         # TB/s; TFLOP/s chia TB/s ra FLOP/byte
DIEM_GAY = DINH / BW
theo_sach("điểm gãy ở FP16 (FLOP/byte)", DIEM_GAY, sach=153, nguon="2.04 TB/s · 153 FLOP/byte")
print(f"cùng băng thông, điểm gãy với đỉnh FP32: {DINH_FP32 / BW:.2f} FLOP/byte")


def mai(mat_do, dinh=DINH, bw=BW):
    """Trần roofline: hiệu năng tối đa ở một mật độ tính toán."""
    return np.minimum(dinh, bw * mat_do)

### Dựng BERT-Base để đếm

Phép tính nhẩm 1.2 cho BERT-Base 110 triệu tham số, tức 220 MB ở FP16, và khoảng 22 GFLOP cho một lượt
xuôi với độ dài chuỗi 128. Ô dưới dựng BERT-Base bằng các lớp có sẵn của PyTorch trên thiết bị `meta`,
loại thiết bị chỉ giữ hình dạng tensor, không cấp phát bộ nhớ và không tính gì, rồi đếm cả hai con số
từ hình dạng các tầng. Cấu hình (12 tầng, chiều 768, 12 đầu tập trung, tầng ẩn 3,072, từ vựng 30,522) là
cấu hình công bố của BERT-Base, không phải số của chương: chương chỉ cho tổng, nên tổng là thứ được kiểm.

**Dự đoán:** 2 × 110 triệu × 128 bằng 28 GFLOP, không phải 22. Phần chênh nằm ở đâu?

In [ ]:
import torch.nn as nn

D, TANG, DAU, FF, TU_VUNG, VI_TRI = 768, 12, 12, 3_072, 30_522, 512
with torch.device("meta"):
    ma_hoa = nn.TransformerEncoder(nn.TransformerEncoderLayer(D, DAU, FF, dropout=0.0, batch_first=True),
                                   TANG, enable_nested_tensor=False)
    # ba bảng embedding (từ, vị trí, loại câu), một tầng chuẩn hoá, và tầng pooler ở cuối
    phu = nn.ModuleList([nn.Embedding(TU_VUNG, D), nn.Embedding(VI_TRI, D), nn.Embedding(2, D),
                         nn.LayerNorm(D), nn.Linear(D, D)])
n_tham_so = sum(p.numel() for m in (ma_hoa, phu) for p in m.parameters())
n_nhung = sum(p.numel() for m in list(phu)[:3] for p in m.parameters())
byte_trong_so = 2 * n_tham_so  # FP16: 2 byte mỗi tham số
print(f"tham số: {n_tham_so:,}, trong đó {n_nhung:,} nằm ở ba bảng embedding")
theo_sach("tham số (triệu)", n_tham_so / 1e6, sach=110, nguon="110 triệu · 220 MB", tol=1)
theo_sach("trọng số ở FP16 (MB)", byte_trong_so / 1e6, sach=220, nguon="110 triệu · 220 MB", tol=1.5)

# số trọng số tham gia phép nhân: mọi tầng tuyến tính, cộng phép chiếu Q, K, V của khối tự tập trung
w_nhan = sum(m.weight.numel() for m in ma_hoa.modules() if isinstance(m, nn.Linear))
w_nhan += sum(m.in_proj_weight.numel() for m in ma_hoa.modules() if isinstance(m, nn.MultiheadAttention))


def flop(b, S):
    """FLOP một lượt xuôi qua 12 tầng mã hoá, 2 FLOP cho mỗi phép nhân-cộng."""
    tuyen_tinh = S * w_nhan        # mỗi token nhân với mọi ma trận trọng số một lần
    chu_y = TANG * 2 * S * S * D   # QKᵀ rồi nhân với V: mỗi tích S·S·D phép nhân-cộng mỗi tầng
    return 2 * b * (tuyen_tinh + chu_y)


theo_sach("GFLOP một lượt xuôi, batch 1", flop(1, 128) / 1e9, sach=22, nguon="≈22 GFLOP · S = 128", tol=0.5)
print(f"trọng số tham gia phép nhân: {w_nhan:,}; phần tập trung chiếm"
      f" {100 * TANG * 4 * 128**2 * D / flop(1, 128):.1f} % số FLOP ở S = 128")

Phần chênh nằm ở ba bảng embedding (embedding table), gần 24 triệu tham số. Một token chỉ **tra** một hàng
của bảng, không nhân với cả bảng, nên các tham số ấy chiếm byte mà không tốn FLOP. Ở S = 128, phần tính ma
trận tập trung chỉ chiếm vài phần trăm; phần lớn số FLOP là các phép nhân ma trận với trọng số.

Giờ là phép tính nhẩm của sách, với các số đã làm tròn của sách. Mô hình chỉ tính trọng số giả định mỗi
lượt suy luận phải nạp toàn bộ 220 MB trọng số từ bộ nhớ, và bỏ qua byte của kích hoạt.

In [ ]:
flop_sach = sach(22, nguon="≈22 GFLOP", tol=0.5) * 1e9
byte_sach = sach(220, nguon="110 triệu · 220 MB") * 1e6
I1 = flop_sach / byte_sach
theo_sach("mật độ tính toán ở batch 1 (FLOP/byte)", I1, sach=100, nguon="S = 128 · 100 FLOP/byte")
assert I1 < DIEM_GAY  # dưới điểm gãy: nghẽn ở bộ nhớ, hiệu năng nằm trên sườn băng thông
hieu_nang_1 = float(mai(I1))
print(f"hiệu năng đạt được: {I1:.0f} × {BW} = {hieu_nang_1:.1f} TFLOP/s;"
      f" sách in {sach(203.9, nguon='203.9 TFLOP/s')} TFLOP/s")
theo_sach("mức sử dụng GPU ở batch 1 (%)", 100 * hieu_nang_1 / DINH, sach=65.4, nguon="203.9 TFLOP/s · 65.4 phần trăm")
I_dem = flop(1, 128) / byte_trong_so
print(f"với số đếm chưa làm tròn ở trên: {I_dem:.1f} FLOP/byte, mức sử dụng {100 * mai(I_dem) / DINH:.1f} %")

B32 = sach(32, nguon="32 · 704 GFLOP")
theo_sach("GFLOP ở batch 32", B32 * flop_sach / 1e9, sach=704, nguon="32 · 704 GFLOP")
I32 = B32 * flop_sach / byte_sach  # cùng 220 MB trọng số, gấp 32 lần tính toán
theo_sach("mật độ tính toán ở batch 32 (FLOP/byte)", I32, sach=3_200, nguon="704 GFLOP · 3,200 FLOP/byte")
assert I32 > DIEM_GAY
hieu_suat = sach(85, nguon="3,200 FLOP/byte · 85 phần trăm") / 100  # giả định của sách về bản hiện thực
theo_sach("hiệu năng ở batch 32 (TFLOP/s)", hieu_suat * DINH, sach=265.2, nguon="85 phần trăm · 265.2 TFLOP/s")

100 × 2.04 cho 204.0, còn sách in 203.9. Hai con số lệch nhau ở chữ số cuối, có thể vì sách tính từ một
băng thông chưa làm tròn tới 2.04; mức sử dụng làm tròn ra cùng 65.4 phần trăm theo cả hai cách. Sách
cũng nói rõ con số 85 phần trăm là một giả định về hiệu suất của bản hiện thực, không phải dự báo của
roofline: roofline chỉ nói batch 32 không còn bị băng thông chặn.

### Mật độ tính toán theo batch và độ dài chuỗi

Trong mô hình chỉ tính trọng số, byte cố định ở 220 MB còn FLOP tăng theo batch, nên mật độ tăng tuyến
tính và vượt điểm gãy rất sớm. Nhưng mỗi chuỗi trong batch còn ghi và đọc lại kích hoạt của nó. Đường
nét đứt dưới đây thêm phần ấy, theo một **giả định của sổ tay**: mỗi tầng ghi hoặc đọc `K_KICH_HOAT`
tensor cỡ S×768, và ma trận tập trung cỡ 12×S×S được ghi rồi đọc lại hai lần, trước và sau softmax.

**Dự đoán:** khi tính cả kích hoạt, mật độ còn tăng mãi theo batch không?

In [ ]:
K_KICH_HOAT = 16  # giả định: số tensor cỡ S×D mỗi tầng ghi hoặc đọc, khi các kernel không được hợp nhất


def byte_doc_ghi(b, S, kich_hoat=False):
    """Byte đọc ghi trong một lượt xuôi: trọng số FP16 nạp một lần cho cả batch, cộng kích hoạt nếu muốn."""
    tong = byte_trong_so
    if kich_hoat:
        tong += 2 * b * TANG * (K_KICH_HOAT * S * D + 4 * DAU * S * S)
    return tong


cac_b = 2 ** np.arange(0, 9)  # batch 1 tới 256
cac_S = [64, 128, 256, 512]
for S in cac_S:
    for kh in (False, True):
        I = np.array([flop(b, S) / byte_doc_ghi(b, S, kh) for b in cac_b])
        vuot = cac_b[I > DIEM_GAY]
        chu = f"batch {vuot[0]}" if len(vuot) else "không batch nào tới 256"
        tran = f", tiến tới {flop(1, S) / (byte_doc_ghi(1, S, True) - byte_trong_so):.0f} FLOP/byte" if kh else ""
        print(f"S = {S:>3}, {'có kích hoạt' if kh else 'chỉ trọng số'}: vượt điểm gãy từ {chu}{tran}")

RESNET = sach(300, nguon="≈300 · ≈100 FLOP/byte", tol=50)
print(f"để so: một lượt xuôi ResNet-50 minh hoạ ở batch lớn có mật độ khoảng {RESNET} FLOP/byte,"
      f" {'trên' if RESNET > DIEM_GAY else 'dưới'} điểm gãy")
MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for S, mau in zip(cac_S, MAU):
    for kh, kieu in ((False, "-"), (True, "--")):
        I = np.array([flop(b, S) / byte_doc_ghi(b, S, kh) for b in cac_b])
        nhan = f"S = {S}" + (", có kích hoạt (giả định)" if kh else ", chỉ trọng số")
        ax1.loglog(cac_b, I, kieu, color=mau, lw=1.8, label=nhan)
        ax2.semilogx(cac_b, 100 * mai(I) / DINH, kieu, color=mau, lw=1.8)
ax1.axhline(DIEM_GAY, color="black", lw=1, ls=":")
ax1.text(256, DIEM_GAY * 0.8, f"điểm gãy A100: {DIEM_GAY:.0f}", fontsize=8, ha="right", va="top")
ax1.plot([1, B32], [I1, I32], "ko", ms=5, zorder=5)
ax1.annotate("sách: 100", (1, I1), xytext=(6, -12), textcoords="offset points", fontsize=8)
ax1.annotate("sách: 3,200", (B32, I32), xytext=(-8, 6), textcoords="offset points", fontsize=8, ha="right")
ax1.set_ylabel("mật độ tính toán (FLOP/byte)")
ax1.set_title("Mật độ tính toán của BERT-Base")
ax2.plot(1, 100 * hieu_nang_1 / DINH, "ko", ms=5)
ax2.annotate("sách: 65.4 %", (1, 100 * hieu_nang_1 / DINH), xytext=(8, -4), textcoords="offset points", fontsize=8)
ax2.axhline(100 * hieu_suat, color="0.4", lw=1, ls=":")
ax2.text(256, 100 * hieu_suat - 6, "85 %: giả định của sách cho bản hiện thực", fontsize=8, ha="right", color="0.3")
ax2.set_ylim(0, 108)
ax2.set_ylabel("trần mức sử dụng theo roofline (%)")
ax2.set_title("Mức sử dụng mà roofline cho phép")
for ax in (ax1, ax2):
    ax.set_xticks(cac_b, [str(b) for b in cac_b])
    ax.set_xlabel("kích thước batch")
fig.legend(*ax1.get_legend_handles_labels(), loc="lower center", ncol=4, fontsize=8, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

Chỉ tính trọng số, batch 2 đã vượt điểm gãy ở S = 128, và mật độ tăng mãi theo batch. Khi tính cả kích hoạt,
phần byte của mỗi chuỗi tăng cùng với FLOP của nó, nên mật độ dừng lại ở một mức trần: FLOP của một chuỗi
chia cho byte kích hoạt của nó. Chuỗi càng dài thì ma trận tập trung, tỉ lệ với S², càng nặng, nên mức trần
ấy càng thấp. Kết luận của sách vẫn đứng với BERT, nhưng mô hình chỉ tính trọng số lạc quan dần khi
batch lớn lên, và đó là lý do sách ghi rõ nó là mô hình chỉ tính trọng số.

Gom batch nâng mật độ tính toán nhưng mỗi yêu cầu phải chờ gom đủ batch. Đó là đánh đổi giữa thông lượng
(throughput) và độ trễ mà sổ tay 02 đã đo, giữa kịch bản SingleStream và Offline.

## 3. Roofline của CPU của bạn

Đỉnh của tờ thông số chỉ là một nửa của roofline. Ở đây cả hai trần đều được **đo**: băng thông bằng một
phép chép mảng, đỉnh tính toán bằng một phép nhân ma trận (matrix multiplication) lớn. Quy tắc trạng thái
bộ nhớ đệm của góc nhìn 1.4 nói một phép đo DRAM phải vượt khỏi bộ nhớ đệm (cache), nếu không con số đo
được là băng thông của bộ nhớ đệm. Sách đưa cỡ của hai mức trên GPU: khoảng 5 tới 10 TB/s cho bộ nhớ đệm,
khoảng 1 tới 2 TB/s cho DRAM.

**Dự đoán:** khi mảng lớn dần từ 16 KB tới 256 MB, băng thông chép đo được thay đổi thế nào?

In [ ]:
def thoi_gian(f, lap):
    """Trung vị thời gian (giây) của lap lần gọi f(), sau một lần làm nóng."""
    f()
    t = []
    for _ in range(lap):
        t0 = time.perf_counter()
        f()
        t.append(time.perf_counter() - t0)
    return float(np.median(t))


dem_thap = sach(5, nguon="≈5–10 TB/s · ≈1–2 TB/s", tol=0.5)
dram_cao = sach(2, nguon="≈5–10 TB/s · ≈1–2 TB/s", tol=0.5)
print(f"trên GPU của sách, bộ nhớ đệm nhanh hơn DRAM ít nhất {dem_thap / dram_cao:.1f} lần")

cac_co = 2 ** np.arange(14, 29, 2)  # byte mỗi mảng: 16 KB tới 256 MB
bw_chep = []
for co in cac_co:
    nguon_, dich = torch.rand(int(co // 4)), torch.empty(int(co // 4))
    lap = int(max(5, min(2_000, 2**30 // co)))
    bw_chep.append(2 * co / thoi_gian(lambda: dich.copy_(nguon_), lap) / 1e9)  # đọc một mảng, ghi một mảng
    del nguon_, dich
bw_chep = np.array(bw_chep)
lon = torch.rand(2**28 // 4)
bw_doc = lon.numel() * 4 / thoi_gian(lambda: lon.sum(), 5) / 1e9  # chỉ đọc
del lon
BW_DRAM = max(bw_chep[-1], bw_doc)
BW_DEM = bw_chep[:4].max()
do_tren_may("băng thông DRAM (chép hoặc cộng dồn 256 MB, lấy số lớn hơn)", BW_DRAM, "GB/s")
do_tren_may("băng thông cao nhất khi mảng nhỏ, 16 KB tới 1 MB", BW_DEM, "GB/s")
do_tren_may("bộ nhớ đệm nhanh hơn DRAM", BW_DEM / BW_DRAM, "lần")

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.semilogx(cac_co, bw_chep, "o-", color=MAU[0], lw=2)
ax.set_xticks(cac_co, ["16 KB", "64 KB", "256 KB", "1 MB", "4 MB", "16 MB", "64 MB", "256 MB"])
ax.set_ylim(0, bw_chep.max() * 1.15)
ax.set_xlabel("cỡ mỗi mảng")
ax.set_ylabel("GB/s (đo trên máy này)")
ax.set_title("Băng thông của một phép chép, theo cỡ mảng")
plt.tight_layout()
plt.show()

Mảng nhỏ vừa trong bộ nhớ đệm nên chép nhanh hơn; khi mảng lớn dần, con số tụt xuống mức của DRAM. Ở cỡ
nhỏ nhất, một phép chép chỉ mất vài micro-giây, nên phụ trội của chính lần gọi cũng kéo con số xuống. Mái
băng thông DRAM dưới đây lấy số đo ở 256 MB, còn mái của bộ nhớ đệm lấy số cao nhất của các mảng nhỏ.

Trần thứ hai là đỉnh tính toán. CPU không có một con số trên tờ thông số mà sổ tay đọc được, nên trần
ở đây là mức cao nhất đo được của phép nhân ma trận vuông, cỡ 64 tới 2,048. Phép nhân hai ma trận n×n
làm 2n³ FLOP trên ít nhất 3n² số 4 byte, nên mật độ tính toán của nó là n/6 FLOP mỗi byte.

In [ ]:
cac_n = [64, 128, 256, 512, 1_024, 2_048]
gflop_mm = []
for n in cac_n:
    A, Bm = torch.randn(n, n), torch.randn(n, n)
    gflop_mm.append(2 * n**3 / thoi_gian(lambda: A @ Bm, max(3, int(2e9 // (2 * n**3)))) / 1e9)
del A, Bm
gflop_mm = np.array(gflop_mm)
DINH_CPU = gflop_mm.max()
for n, g in zip(cac_n, gflop_mm):
    do_tren_may(f"nhân ma trận {n}×{n}", g, "GFLOP/s")
do_tren_may("điểm gãy của CPU này (trần tính toán chia băng thông DRAM)", DINH_CPU / BW_DRAM, "FLOP/byte")

### Ba kernel dưới mái

Ô dưới đo hai loại kernel nữa.

* **Cộng hai vectơ**, từ 16 tới 16.8 triệu phần tử: mỗi phần tử một FLOP trên 12 byte, nên mật độ chỉ là
  1/12 FLOP mỗi byte ở mọi cỡ. Với vectơ nhỏ, gần như toàn bộ thời gian là phụ trội cố định (fixed
  overhead) của việc điều phối một phép toán qua PyTorch.
* **Nhân một ma trận trọng số 8,192×4,096 (128 MB ở FP32) với một batch vectơ**, batch từ 1 tới 128. Ma
  trận lớn hơn bộ nhớ đệm của hầu hết CPU, nên mỗi lần gọi phải đọc nó từ DRAM, giống BERT nạp trọng số.
  Đây là một tầng tuyến tính khi suy luận, như phần 2.

**Dự đoán:** phép nhân ma trận với một vectơ, batch 1, nằm gần trần nào?

In [ ]:
cac_m = 2 ** np.arange(4, 25, 2)  # số phần tử: 16 tới 16.8 triệu
t_cong = []
for m in cac_m:
    x, y, z = torch.rand(int(m)), torch.rand(int(m)), torch.empty(int(m))
    t_cong.append(thoi_gian(lambda: torch.add(x, y, out=z), int(max(5, min(5_000, 2**26 // m)))))
del x, y, z
t_cong = np.array(t_cong)
do_tren_may("cộng 16 phần tử: một lần gọi", 1e6 * t_cong[0], "µs")
do_tren_may("cộng 16.8 triệu phần tử: băng thông", 12 * cac_m[-1] / t_cong[-1] / 1e9, "GB/s")
do_tren_may("phần phụ trội trong thời gian cộng 16 nghìn phần tử", 100 * min(1, t_cong[0] / t_cong[5]), "%")

K_VAO, N_RA = 8_192, 4_096
W = torch.randn(K_VAO, N_RA)
cac_bv = 2 ** np.arange(0, 8)  # batch 1 tới 128
gflop_tt, mat_do_tt = [], []
for b in cac_bv:
    xb = torch.randn(int(b), K_VAO)
    t = thoi_gian(lambda: torch.mm(xb, W), 3)
    gflop_tt.append(2 * b * K_VAO * N_RA / t / 1e9)
    mat_do_tt.append(2 * b * K_VAO * N_RA / (4 * (K_VAO * N_RA + b * K_VAO + b * N_RA)))
del W, xb
gflop_tt, mat_do_tt = np.array(gflop_tt), np.array(mat_do_tt)
for b, g, i in zip(cac_bv, gflop_tt, mat_do_tt):
    do_tren_may(f"tầng tuyến tính 128 MB, batch {b:>3} (mật độ {i:.1f} FLOP/byte)", g, "GFLOP/s")

In [ ]:
I_truc = np.logspace(-2, 3, 200)
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.loglog(I_truc, mai(I_truc, DINH_CPU, BW_DRAM), color="black", lw=2, label="mái: băng thông DRAM")
ax.loglog(I_truc, mai(I_truc, DINH_CPU, BW_DEM), color="black", lw=1, ls="--", label="mái: băng thông bộ nhớ đệm")
ax.loglog(np.full(len(cac_m), 1 / 12), cac_m / t_cong / 1e9, "s", color=MAU[1], ms=6,
          label="cộng hai vectơ, 16 tới 16.8 triệu phần tử")
ax.loglog(mat_do_tt, gflop_tt, "o-", color=MAU[0], ms=6, label="tầng tuyến tính 128 MB, batch 1 tới 128")
ax.loglog(np.array(cac_n) / 6, gflop_mm, "^-", color=MAU[2], ms=6, label="nhân ma trận vuông, 64 tới 2,048")
ax.annotate("16 phần tử", (1 / 12, cac_m[0] / t_cong[0] / 1e9), xytext=(8, 0), textcoords="offset points",
            fontsize=8, va="center")
ax.annotate("16.8 triệu phần tử", (1 / 12, cac_m[-1] / t_cong[-1] / 1e9), xytext=(28, -22),
            textcoords="offset points", fontsize=8, va="top", arrowprops={"arrowstyle": "-", "color": "0.4"})
ax.annotate("batch 1", (mat_do_tt[0], gflop_tt[0]), xytext=(0, -9), textcoords="offset points", fontsize=8,
            ha="center", va="top")
ax.set_xlim(1e-2, 1e3)
ax.set_xlabel("mật độ tính toán (FLOP/byte)")
ax.set_ylabel("GFLOP/s (đo trên máy này)")
ax.set_title(f"Roofline đo trên CPU này, {torch.get_num_threads()} luồng")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

print("kernel                              % đỉnh tính toán   % mái ở mật độ của nó")
for ten, i, g in (("cộng 16 phần tử", 1 / 12, cac_m[0] / t_cong[0] / 1e9),
                  ("cộng 16.8 triệu phần tử", 1 / 12, cac_m[-1] / t_cong[-1] / 1e9),
                  ("tầng tuyến tính, batch 1", mat_do_tt[0], gflop_tt[0]),
                  ("tầng tuyến tính, batch 128", mat_do_tt[-1], gflop_tt[-1])):
    print(f"{ten:<34}  {100 * g / DINH_CPU:>12.2f}   {100 * g / mai(i, DINH_CPU, BW_DRAM):>16.1f}  (đo trên máy này)")

Đọc bảng theo hai cột. Phép cộng nhỏ và phép nhân với một vectơ đều dùng một phần nhỏ đỉnh tính toán,
tức MFU của cả hai đều thấp, nhưng vì hai lý do khác nhau:

* phép nhân với một vectơ đạt một phần lớn của mái băng thông: nó nghẽn ở bộ nhớ, và chỉ tăng batch, tức
  tăng mật độ, mới đưa nó lên;
* phép cộng 16 phần tử nằm dưới **cả hai** mái rất xa: cả FLOP lẫn byte đều không phải giới hạn. Đó là cái
  bẫy mức sử dụng của góc nhìn 1.5, khi kernel quá nhỏ và thời gian nằm ở khoảng trống giữa các lần chạy.

Đó là câu trả lời cho checkpoint 1.1: riêng một tỉ lệ MFU không chỉ ra điểm nghẽn, phải đặt nó lên
roofline. Đường cong băng thông theo cỡ mảng cũng cho thấy vì sao quy tắc trạng thái bộ nhớ đệm tồn tại:
một phép đo "DRAM" trên mảng nhỏ báo băng thông của bộ nhớ đệm.

## 4. Hiệu suất mở rộng

Phép tính nhẩm 1.3 đặt một nhóm huấn luyện ResNet-50 trên ImageNet: 24 giờ với 1 GPU, 4 giờ với 8 GPU.
Phương trình 3 định nghĩa hiệu suất mở rộng mạnh, khi bài toán giữ nguyên còn bộ xử lý tăng lên:

$$\text{Eff}_{\text{scaling}}(N) = \frac{T(1)}{N \cdot T(N)}$$

**Dự đoán:** 8 GPU nhanh gấp 6 lần một GPU. Hiệu suất là bao nhiêu?

In [ ]:
T1 = sach(24, nguon="24 giờ · 8 GPU · 4 giờ")
N8 = sach(8, nguon="24 giờ · 8 GPU · 4 giờ")
T8 = sach(4, nguon="24 giờ · 8 GPU · 4 giờ")
hieu_suat_8 = T1 / (N8 * T8)
theo_sach("hiệu suất mở rộng ở 8 GPU (%)", 100 * hieu_suat_8, sach=75, nguon="4 giờ · 75 phần trăm")
theo_sach("thời gian nếu mở rộng hoàn hảo (giờ)", T1 / N8, sach=3, nguon="75 phần trăm · 3 giờ")
theo_sach("phần hiệu suất bị mất (%)", 100 * (1 - hieu_suat_8), sach=25, nguon="3 giờ · 25 phần trăm")

# bảng 7: bốn nguồn mất hiệu suất, (thấp, cao) theo phần trăm; sách ghi đây là phân rã minh hoạ
bang7 = {
    "đồng bộ gradient": (sach(10, nguon="10–15 · 3–5"), sach(15, nguon="10–15 · 3–5")),
    "sao chép CPU ↔ GPU": (sach(3, nguon="10–15 · 3–5"), sach(5, nguon="10–15 · 3–5")),
    "mất cân bằng tải": (sach(2, nguon="3–5 · 2–5 · 2–5"), sach(5, nguon="3–5 · 2–5 · 2–5")),
    "hiệu ứng kích thước batch": (sach(2, nguon="2–5 · 2–5 phần trăm"), sach(5, nguon="2–5 · 2–5 phần trăm")),
}
thap, cao = sum(v[0] for v in bang7.values()), sum(v[1] for v in bang7.values())
print(f"bốn nguồn cộng lại: {thap} tới {cao} %, quanh phần {100 * (1 - hieu_suat_8):.0f} % bị mất")
assert thap <= 100 * (1 - hieu_suat_8) <= cao

Sách thêm một so sánh: hệ thống đạt thông lượng 2 lần ở hiệu suất 50 phần trăm có thể kém hơn hệ thống đạt
1.5 lần ở hiệu suất 90 phần trăm, tuỳ ràng buộc chi phí. Thông lượng gấp $S$ lần ở hiệu suất $E$ nghĩa là
dùng $S/E$ GPU, và mỗi việc tốn $1/E$ lần số GPU-giờ của một GPU.

In [ ]:
tang_a, hs_a = sach(2, nguon="2× ở 50 phần trăm"), sach(50, nguon="2× ở 50 phần trăm") / 100
tang_b, hs_b = sach(1.5, nguon="1.5× ở 90 phần trăm"), sach(90, nguon="1.5× ở 90 phần trăm") / 100
for ten, tang, hs in (("hệ thống A", tang_a, hs_a), ("hệ thống B", tang_b, hs_b)):
    print(f"{ten}: nhanh {tang} lần ở hiệu suất {100 * hs:.0f} %, tức {tang / hs:.2f} GPU,"
          f" {1 / hs:.2f} GPU-giờ cho mỗi GPU-giờ của một GPU")
print(f"A xong sớm hơn B {tang_a / tang_b:.2f} lần, nhưng tốn gấp {(1 / hs_a) / (1 / hs_b):.2f} lần GPU-giờ")

### Ngoại suy từ 8 GPU

Cạm bẫy thứ hai khi đo huấn luyện mà sách nêu là ngoại suy tuyến tính: giả định 75 phần trăm giữ nguyên ở
mọi quy mô. Một mô hình đơn giản tách thời gian huấn luyện thành phần chia được cho $N$ GPU và phần không
chia được, như truyền thông:

$$T(N) = T(1)\left(s + \frac{1-s}{N}\right)$$

Đây là định luật Amdahl, với $s$ là phần không chia được. Phép AllReduce dạng vòng cũng cho dạng này, vì
lượng dữ liệu mỗi GPU gửi đi gần như không đổi theo $N$. Mô hình là **giả định của sổ tay**; ô dưới
chỉnh $s$ để nó đi qua đúng điểm 8 GPU của sách.

**Dự đoán:** ở 64 GPU, hiệu suất còn bao nhiêu?

In [ ]:
s_noi_tiep = (T8 / T1 - 1 / N8) / (1 - 1 / N8)  # chọn s sao cho T(8) = 4 giờ
print(f"phần không chia được: s = {s_noi_tiep:.4f} (tức 1/{1 / s_noi_tiep:.0f})")


def t_mo_rong(N):
    return T1 * (s_noi_tiep + (1 - s_noi_tiep) / N)


assert math.isclose(t_mo_rong(N8), T8)
for N in (8, 64, 512):
    print(f"{N:>4} GPU: mô hình cho {t_mo_rong(N):.2f} giờ, hiệu suất {100 * T1 / (N * t_mo_rong(N)):.0f} %;"
          f" ngoại suy tuyến tính ở 75 % cho {T1 / (N * hieu_suat_8):.2f} giờ")

cac_N = 2 ** np.arange(0, 11)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.semilogx(cac_N, 100 * T1 / (cac_N * t_mo_rong(cac_N)), "o-", color=MAU[0], lw=2, ms=4,
             label="mô hình phần không chia được (giả định)")
ax1.semilogx(cac_N[3:], np.full(len(cac_N) - 3, 100 * hieu_suat_8), "--", color=MAU[1], lw=2,
             label="ngoại suy tuyến tính: 75 % mãi mãi")
ax1.plot(N8, 100 * hieu_suat_8, "ko", ms=6, zorder=5)
ax1.annotate("sách: 8 GPU, 75 %", (N8, 100 * hieu_suat_8), xytext=(-8, -16), textcoords="offset points",
             fontsize=8, ha="right")
ax1.set_ylim(0, 105)
ax1.set_ylabel("hiệu suất mở rộng (%)")
ax2.loglog(cac_N, t_mo_rong(cac_N), "o-", color=MAU[0], lw=2, ms=4)
ax2.loglog(cac_N[3:], T1 / (cac_N[3:] * hieu_suat_8), "--", color=MAU[1], lw=2)
ax2.loglog(cac_N, T1 / cac_N, ":", color="0.4", lw=1.5, label="mở rộng hoàn hảo")
ax2.plot(N8, T8, "ko", ms=6, zorder=5)
ax2.set_ylabel("thời gian huấn luyện (giờ)")
for ax in (ax1, ax2):
    ax.set_xticks(cac_N, [str(n) for n in cac_N], fontsize=8)
    ax.set_xlabel("số GPU")
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
fig.legend(h1 + h2, l1 + l2, loc="lower center", ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Từ một điểm 8 GPU: hai cách ngoại suy (tính từ số của sách)", fontsize=10)
plt.tight_layout(rect=(0, 0.07, 1, 1))
plt.show()

Một phần không chia được cỡ 1/21 thời gian đã đủ kéo hiệu suất từ 75 phần trăm ở 8 GPU xuống còn một phần
tư ở 64 GPU, trong khi ngoại suy tuyến tính hứa 64 GPU sẽ xong trong nửa giờ. Thời gian không bao giờ
xuống dưới $s \cdot T(1)$, dù thêm bao nhiêu GPU. Trong thực tế phần truyền thông còn có thể tăng theo $N$,
vì độ trễ của mỗi lượt gửi cộng dồn, và khi đó đường cong còn xấu hơn. Đó là lý do sách yêu cầu đo hiệu
suất mở rộng ở từng quy mô, thay vì suy ra từ một quy mô nhỏ.

## 5. Thời gian đạt độ chính xác

Phương trình 1 định nghĩa thời gian huấn luyện là thời điểm sớm nhất độ chính xác đạt mục tiêu:

$$T_{\text{train}} = \inf\{t \ge 0 : \text{Accuracy}(t) \ge \text{mục tiêu}\}$$

Một lần chạy không bao giờ đạt mục tiêu thì không có thời gian hữu hạn, nên tốc độ thô không thể thưởng
cho hệ thống trượt tiêu chí chất lượng. MLPerf đặt ResNet-50 trên ImageNet ở mục tiêu 75.9 phần trăm
top-1.

Ô dưới huấn luyện cùng một mạng 784-256-10 trên 20,000 ảnh MNIST với bốn cấu hình kích thước batch (batch
size) và tốc độ học (learning rate), mỗi cấu hình với ba hạt giống, tối đa 5 epoch, bằng SGD có động lượng
(momentum) 0.9. Mục tiêu là 96 phần trăm trên 5,000 ảnh kiểm định tách riêng, một **giả định của sổ
tay**. Đồng hồ chỉ chạy trong các bước huấn luyện; thời gian chấm điểm, mỗi 5,000 mẫu một lần, không được
tính.

**Dự đoán:** cấu hình có thông lượng cao nhất có chạm mục tiêu sớm nhất không?

In [ ]:
import torchvision

muc_tieu_mlperf = sach(75.9, nguon="75.9 phần trăm")
tap = torchvision.datasets.MNIST(DATA, train=True, download=True)
X = tap.data.reshape(-1, 784).float() / 255
X = (X - X.mean()) / X.std()
hoan_vi = torch.randperm(len(X), generator=torch.Generator().manual_seed(SEED))
X_hl, Y_hl = X[hoan_vi[:20_000]], tap.targets[hoan_vi[:20_000]]
X_kd, Y_kd = X[hoan_vi[-5_000:]], tap.targets[hoan_vi[-5_000:]]
MUC_TIEU = 0.96  # giả định của sổ tay, trên tập kiểm định
print(f"huấn luyện {len(X_hl):,} ảnh, kiểm định {len(X_kd):,} ảnh; mục tiêu {100 * MUC_TIEU:.0f} %"
      f" (MLPerf đặt {muc_tieu_mlperf} % cho ResNet-50 trên ImageNet)")

In [ ]:
CAU_HINH = {  # tên: (kích thước batch, tốc độ học)
    "A: batch 32, học 0.01": (32, 0.01),
    "B: batch 128, học 0.05": (128, 0.05),
    "C: batch 1024, học 0.05": (1_024, 0.05),
    "D: batch 1024, học 0.2": (1_024, 0.2),
}
HAT_GIONG = (1, 2, 3)


def huan_luyen(bs, lr, hat, so_epoch=5, cham_moi=5_000):
    """Một lần chạy. Trả về thời gian đạt mục tiêu (vô hạn nếu không đạt), thông lượng và nhật ký."""
    torch.manual_seed(hat)
    mo_hinh = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))
    toi_uu = torch.optim.SGD(mo_hinh.parameters(), lr=lr, momentum=0.9)
    sinh = torch.Generator().manual_seed(hat)
    dong_ho, mau, moc, nhat_ky, tta = 0.0, 0, cham_moi, [], math.inf
    for _ in range(so_epoch):
        thu_tu = torch.randperm(len(X_hl), generator=sinh)
        for i in range(0, len(X_hl), bs):
            chon = thu_tu[i:i + bs]
            t0 = time.perf_counter()
            toi_uu.zero_grad()
            nn.functional.cross_entropy(mo_hinh(X_hl[chon]), Y_hl[chon]).backward()
            toi_uu.step()
            dong_ho += time.perf_counter() - t0
            mau += len(chon)
            if mau >= moc:  # chấm điểm ngoài đồng hồ
                moc += cham_moi
                with torch.no_grad():
                    dung = (mo_hinh(X_kd).argmax(1) == Y_kd).float().mean().item()
                nhat_ky.append((dong_ho, dung))
                if dung >= MUC_TIEU and tta == math.inf:
                    tta = dong_ho
    return tta, mau / dong_ho, np.array(nhat_ky)


ket_qua = {ten: [huan_luyen(bs, lr, h) for h in HAT_GIONG] for ten, (bs, lr) in CAU_HINH.items()}
for ten, chay in ket_qua.items():
    tta = [r[0] for r in chay]
    chu = ", ".join("không đạt" if math.isinf(t) else f"{t:.2f} s" for t in tta)
    do_tren_may(f"{ten}: thông lượng", np.median([r[1] for r in chay]), "mẫu/giây")
    print(f"   thời gian đạt {100 * MUC_TIEU:.0f} % với ba hạt giống: {chu};"
          f" độ chính xác cuối cao nhất {100 * max(r[2][-1, 1] for r in chay):.2f} %")
thong_luong = {k: np.median([r[1] for r in v]) for k, v in ket_qua.items()}
tta_tv = {k: np.median([r[0] for r in v]) for k, v in ket_qua.items()}  # trung vị; vô hạn nếu đa số không đạt
print("xếp theo thông lượng, cao trước:", ", ".join(k[0] for k in sorted(thong_luong, key=lambda k: -thong_luong[k])))
dat = sorted((k for k in tta_tv if not math.isinf(tta_tv[k])), key=tta_tv.get)
khong = [k[0] for k in tta_tv if math.isinf(tta_tv[k])]
print("xếp theo trung vị thời gian đạt độ chính xác, sớm trước:", ", ".join(k[0] for k in dat)
      + (f"; không đạt: {', '.join(khong)}" if khong else ""))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.4))
for (ten, chay), mau in zip(ket_qua.items(), MAU):
    for j, (tta, _, nhat_ky) in enumerate(chay):
        ax.plot(nhat_ky[:, 0], 100 * nhat_ky[:, 1], "-", color=mau, lw=1.6, alpha=1 if j == 0 else 0.45,
                label=ten if j == 0 else None)
        if not math.isinf(tta):
            ax.plot(tta, 100 * nhat_ky[nhat_ky[:, 0] == tta, 1][0], "o", color=mau, ms=5)
ax.axhline(100 * MUC_TIEU, color="black", lw=1, ls=":")
ax.text(0.01, 100 * MUC_TIEU + 0.15, f"mục tiêu {100 * MUC_TIEU:.0f} % (giả định)", fontsize=8, ha="left",
        transform=ax.get_yaxis_transform())
ax.set_ylim(85, 100)
ax.set_xlabel("thời gian huấn luyện, giây (đo trên máy này)")
ax.set_ylabel("độ chính xác kiểm định (%)")
ax.set_title("Bốn cấu hình, ba hạt giống mỗi cấu hình; chấm tròn: lúc chạm mục tiêu")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=4, fontsize=8)
plt.tight_layout()
plt.show()

So hai dòng xếp hạng. Batch 1024 thường cho thông lượng cao nhất, vì phụ trội của mỗi bước được chia cho
nhiều mẫu hơn. Nhưng cùng tốc độ học, batch lớn làm ít bước cập nhật hơn trên mỗi epoch, nên cần nhiều mẫu
hơn để tới cùng một độ chính xác, và có thể không tới trong ngân sách. Tăng tốc độ học có thể bù lại một
phần; hãy xem ba hạt giống của D nằm gần mục tiêu đến đâu. Một lần chạy không chạm mục tiêu có thời gian
vô hạn, dù thông lượng của nó cao đến đâu.

Ba hạt giống của cùng một cấu hình cũng không cho cùng một thời gian. Đó là lý do MLPerf Training nhận
nhiều lần chạy để mô tả biến thiên ngẫu nhiên, và một báo cáo đáng tin phải báo phân phối của các lần chạy,
thay vì chỉ lần chạy tốt nhất. Sách nêu cùng cạm bẫy với TF32: nhanh hơn mỗi bước mà vẫn thua tổng thể nếu
phải chạy nhiều vòng lặp hơn mới đạt mục tiêu.

## Thử thêm

1. Ở phần 2, đổi `K_KICH_HOAT` thành 32. Ở S = 512, batch nào còn vượt được điểm gãy?
2. Ở phần 4, giả sử đo được thêm một điểm: 64 GPU mất 1 giờ. Hiệu suất ở 64 GPU là bao nhiêu, và mô hình
   phần không chia được có còn đi qua cả hai điểm không?
3. Ở phần 5, đổi `MUC_TIEU` thành 0.95 rồi 0.97. Thứ tự xếp theo thời gian đạt độ chính xác có đổi không?

## Tóm lại

* Đỉnh của tờ thông số là trần, không phải dự báo: 312 TFLOP/s ở MFU 30 tới 50 phần trăm chỉ còn 93.6 tới
  156 TFLOP/s. Một tỉ lệ MFU đơn lẻ không chỉ ra điểm nghẽn; phải đặt nó lên roofline.
* BERT-Base ở batch 1 có mật độ 100 FLOP mỗi byte, dưới điểm gãy 153 của A100, nên nghẽn ở bộ nhớ ở mức 65.4
  phần trăm. Gom batch đẩy nó qua điểm gãy, nhưng mô hình chỉ tính trọng số lạc quan dần khi batch lớn.
* Roofline đo được của một CPU có cùng hình dạng: một tầng tuyến tính ở batch 1 nghẽn ở băng thông, một kernel
  nhỏ thì nằm dưới cả hai mái vì phụ trội điều phối, và mảng nhỏ báo băng thông của bộ nhớ đệm, không phải DRAM.
* 24 giờ trên 1 GPU và 4 giờ trên 8 GPU là hiệu suất 75 phần trăm. Một phần không chia được cỡ 1/21 kéo hiệu
  suất xuống một phần tư ở 64 GPU, nên không được ngoại suy tuyến tính.
* Thông lượng phải đọc cùng thời gian đạt độ chính xác: cấu hình nhanh nhất mỗi mẫu có thể chạm mục tiêu
  muộn hơn, hoặc không bao giờ.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Đo chuẩn hiệu năng — khi lời hứa tối ưu gặp số đo](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch12-benchmarking/), dựng từ chương
[*Benchmarking*](https://mlsysbook.ai/vol1/benchmarking/benchmarking.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.